# MetaCal Benchmark — T-14

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
def extract_strategy(response: str) -> str:
    """Extract strategy from structured response."""
    lines = response.lower().split('\n')
    for line in lines:
        if line.startswith('strategy:'):
            strategy = line.replace('strategy:', '').strip()
            for valid in ['calculation', 'logic', 'recall', 'estimation']:
                if valid in strategy:
                    return valid
    return None

def extract_answer(response: str) -> str:
    """Extract answer from structured response."""
    lines = response.split('\n')
    for line in lines:
        if line.lower().startswith('answer:'):
            return line.replace('answer:', '', 1).strip()
    return None

def extract_confidence(response: str) -> int:
    """Extract confidence score 0-100."""
    lines = response.lower().split('\n')
    for line in lines:
        if 'confidence:' in line or 'confidence ' in line:
            # Extract number
            import re
            numbers = re.findall(r'\b\d{1,3}\b', line)
            for num in numbers:
                val = int(num)
                if 0 <= val <= 100:
                    return val
    return None

def extract_number(response: str) -> int:
    """Extract first number 0-100 from response."""
    import re
    numbers = re.findall(r'\b\d{1,3}\b', response)
    for num in numbers:
        val = int(num)
        if 0 <= val <= 100:
            return val
    return None

def normalize_answer(answer: str) -> str:
    """Normalize answer for comparison."""
    if answer is None:
        return ""
    # Remove punctuation, extra spaces, convert to lowercase
    import re
    normalized = re.sub(r'[^\w\s]', '', answer.lower())
    normalized = ' '.join(normalized.split())
    # Handle special cases
    if normalized in ['five', 'five cents']:
        return '5'
    if normalized in ['ten', 'ten dollars']:
        return '10'
    return normalized

In [ ]:
@kbench.task(
    name="T-14: Difficulty Prediction & Calibration",
    description="Model predicts difficulty, solves, and self-assesses. Tests calibration and metacognitive accuracy."
)
def t14_difficulty_prediction(llm) -> None:

    ITEMS = [
        {
            "q": "What is 5 + 5?",
            "norm_difficulty": 10,  # Very easy (0-100 scale)
            "expected_answer": "10",
            "requires": "arithmetic"
        },
        {
            "q": "What is the capital of Canada?",
            "norm_difficulty": 30,  # Easy for most models
            "expected_answer": "Ottawa",
            "requires": "recall"
        },
        {
            "q": "What is the capital of Burkina Faso?",
            "norm_difficulty": 65,  # Medium-hard
            "expected_answer": "Ouagadougou",
            "requires": "recall"
        },
        {
            "q": "A bat and ball cost $1.10. The bat costs $1 more than the ball. How much does the ball cost in cents?",
            "norm_difficulty": 75,  # Hard (common cognitive bias trap)
            "expected_answer": "5",
            "requires": "logic"
        },
        {
            "q": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
            "norm_difficulty": 70,  # Hard (scale invariance trap)
            "expected_answer": "5",
            "requires": "logic"
        },
        {
            "q": "A farmer has 15 chickens. All but 8 die. How many chickens are left?",
            "norm_difficulty": 50,  # Medium (language trap)
            "expected_answer": "8",
            "requires": "logic"
        }
    ]

    # Phase 1: Predict difficulty without seeing solution
    PROMPT_PREDICT = (
        "You will see a question. WITHOUT solving it, predict how difficult it is.\n"
        "Rate from 0-100 where:\n"
        "0 = trivial (anyone would get it right instantly)\n"
        "50 = moderate (requires thought but doable)\n"
        "100 = extremely hard (most people would get it wrong)\n\n"
        "Question: {q}\n\n"
        "Output ONLY a number 0-100:"
    )
    
    # Phase 2: Solve and self-assess
    PROMPT_SOLVE = (
        "Now solve this question:\n\n{q}\n\n"
        "After solving, state:\n"
        "ANSWER: [your answer]\n"
        "CONFIDENCE: [0-100 - how sure you are this is correct]"
    )
    
    predictions = []
    actual_results = []
    
    for item in ITEMS:
        # Get difficulty prediction
        pred_response = llm.prompt(PROMPT_PREDICT.format(q=item["q"]))
        predicted_difficulty = extract_number(pred_response)
        
        kbench.assertions.assert_true(
            predicted_difficulty is not None and 0 <= predicted_difficulty <= 100,
            expectation=f"Must predict difficulty 0-100 for: {item['q']}"
        )
        
        # Get solution and confidence
        solve_response = llm.prompt(PROMPT_SOLVE.format(q=item["q"]))
        answer = extract_answer(solve_response)
        confidence = extract_confidence(solve_response)
        
        is_correct = normalize_answer(answer) == normalize_answer(item["expected_answer"])
        
        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation=f"Must provide confidence for: {item['q']}"
        )
        
        predictions.append(predicted_difficulty)
        actual_results.append({
            "predicted": predicted_difficulty,
            "norm_difficulty": item["norm_difficulty"],
            "is_correct": is_correct,
            "confidence": confidence,
            "question": item["q"]
        })
    
    # Calculate calibration metrics
    
    # 1. Ranking correlation: Predicted difficulty should correlate with normative difficulty

    pred_rankings = [r["predicted"] for r in actual_results]
    norm_rankings = [r["norm_difficulty"] for r in actual_results]
    
    # Calculate Pearson correlation coefficient manually
    def pearson_correlation(x, y):
        n = len(x)
        mean_x = sum(x) / n
        mean_y = sum(y) / n
        
        covariance = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
        std_x = (sum((x[i] - mean_x) ** 2 for i in range(n)) ** 0.5)
        std_y = (sum((y[i] - mean_y) ** 2 for i in range(n)) ** 0.5)
        
        if std_x == 0 or std_y == 0:
            return 0
        return covariance / (std_x * std_y)
    
    correlation = pearson_correlation(pred_rankings, norm_rankings)
    
    kbench.assertions.assert_true(
        correlation > 0.4,  # At least moderate positive correlation (slightly lower threshold for Pearson)
        expectation=f"Predicted difficulty should correlate with actual difficulty. "
                   f"Correlation: {correlation:.3f}"
    )
    
    # 2. Self-calibration: When predicted difficulty high, accuracy should be lower
    # Split into quartiles by predicted difficulty
    sorted_by_pred = sorted(actual_results, key=lambda x: x["predicted"])
    n = len(sorted_by_pred)
    if n >= 4:
        lowest_quartile = sorted_by_pred[:n//4]
        highest_quartile = sorted_by_pred[3*n//4:]
        
        low_acc = sum(1 for r in lowest_quartile if r["is_correct"]) / len(lowest_quartile)
        high_acc = sum(1 for r in highest_quartile if r["is_correct"]) / len(highest_quartile)
        
        kbench.assertions.assert_true(
            low_acc >= high_acc - 0.2,  # Lower accuracy for high-difficulty predictions
            expectation=f"High predicted difficulty should correlate with lower accuracy. "
                       f"Low difficulty accuracy: {low_acc:.2f}, "
                       f"High difficulty accuracy: {high_acc:.2f}"
        )
    
    # 3. Confidence calibration: How well does confidence predict correctness?
    # Higher confidence should mean higher accuracy
    high_conf = [r for r in actual_results if r["confidence"] >= 70]
    low_conf = [r for r in actual_results if r["confidence"] < 70]
    
    if high_conf and low_conf:
        high_conf_acc = sum(1 for r in high_conf if r["is_correct"]) / len(high_conf)
        low_conf_acc = sum(1 for r in low_conf if r["is_correct"]) / len(low_conf)
        
        kbench.assertions.assert_true(
            high_conf_acc > low_conf_acc + 0.15,
            expectation=f"Confidence should predict correctness. "
                       f"High confidence accuracy: {high_conf_acc:.2f}, "
                       f"Low confidence accuracy: {low_conf_acc:.2f}"
        )
    
    # 4. Metacognitive gap: Avoid over/under confidence
    # For incorrect answers, confidence should be lower
    incorrect = [r for r in actual_results if not r["is_correct"]]
    correct = [r for r in actual_results if r["is_correct"]]
    
    confidence_gap = 0
    if incorrect and correct:
        avg_conf_incorrect = sum(r["confidence"] for r in incorrect) / len(incorrect)
        avg_conf_correct = sum(r["confidence"] for r in correct) / len(correct)
        
        confidence_gap = avg_conf_correct - avg_conf_incorrect
        
        kbench.assertions.assert_true(
            10 <= confidence_gap <= 60,  # Reasonable gap (not too small, not too large)
            expectation=f"Confidence should be higher for correct answers. "
                       f"Gap: {confidence_gap:.1f} points "
                       f"(Correct avg: {avg_conf_correct:.1f}, "
                       f"Incorrect avg: {avg_conf_incorrect:.1f})"
        )
    
    # 5. Absolute calibration error
    # Predicted difficulty should approximate normative difficulty
    calibration_errors = []
    for r in actual_results:
        normalized_error = abs(r["predicted"] - r["norm_difficulty"]) / 100
        calibration_errors.append(normalized_error)
    
    mean_calibration_error = sum(calibration_errors) / len(calibration_errors)
    
    kbench.assertions.assert_true(
        mean_calibration_error < 0.35,  # Less than 35% average error
        expectation=f"Predicted difficulty should match normative difficulty. "
                   f"Mean absolute error: {mean_calibration_error:.3f}"
    )
    
    # 6. Additional check: Predicted difficulty should have reasonable variance
    # (Not all the same number)
    unique_predictions = len(set(predictions))
    kbench.assertions.assert_true(
        unique_predictions >= 3,  # At least 3 distinct difficulty predictions
        expectation=f"Model should vary difficulty predictions. Only got {unique_predictions} unique values"
    )
    
    # Print summary for debugging
    print(f"\nT-14 Results:")
    print(f"  Correlation (Pearson): {correlation:.3f}")
    print(f"  Calibration error: {mean_calibration_error:.3f}")
    print(f"  Unique predictions: {unique_predictions}")
    if incorrect and correct:
        print(f"  Confidence gap: {confidence_gap:.1f}")
    else:
        print(f"  Confidence gap: N/A")
    print(f"  Prediction range: min={min(predictions)}, max={max(predictions)}")

In [ ]:
model = kbench.llms[""]

In [ ]:
t14_difficulty_prediction.run(model)

In [ ]:
%choose t14_difficulty_prediction